# ANNDL Project

Three parts: **classification**, **segmentation**, and **detection** on PASCAL VOC 2012.

Running on local GPU via **Keras 3 + PyTorch backend**.

# 0. Getting started
## Local GPU setup (Keras 3 + PyTorch backend)

In [1]:
import time as _time
import os as _os
_global_start = _time.time()

# Setup Scratch Output Directory to avoid Disk Quota issues
_SCRATCH_DIR = _os.path.join(_os.getcwd(), 'local_test_output')
_OUTPUT_DIR = _SCRATCH_DIR
_os.makedirs(_OUTPUT_DIR, exist_ok=True)
print(f"  📂 Saving mock models locally to: {_OUTPUT_DIR}")

import keras
print("⚠️ LOCAL DRY RUN MODE ACTIVATED ⚠️")
print("=> Overriding Keras Model.fit() to only process 1 batch and 1 epoch!")

if not hasattr(keras.Model, '_original_fit'):
    keras.Model._original_fit = keras.Model.fit
def fast_dry_fit(self, *args, **kwargs):
    kwargs['steps_per_epoch'] = 1
    if 'validation_data' in kwargs:
         kwargs['validation_steps'] = 1
    kwargs['epochs'] = 1
    return keras.Model._original_fit(self, *args, **kwargs)
keras.Model.fit = fast_dry_fit

if not hasattr(keras.Model, '_original_evaluate'):
    keras.Model._original_evaluate = keras.Model.evaluate
def fast_dry_evaluate(self, *args, **kwargs):
    kwargs['steps'] = 1
    return keras.Model._original_evaluate(self, *args, **kwargs)
keras.Model.evaluate = fast_dry_evaluate


def _cell_timer(cell_num):
    elapsed = _time.time() - _global_start
    mins, secs = divmod(int(elapsed), 60)
    hrs, mins = divmod(mins, 60)
    print(f"  ⏱  Total elapsed: {hrs}h {mins:02d}m {secs:02d}s")
    import sys; sys.stdout.flush()

import gc as _gc
import torch as _torch
import json as _json

_HISTORY_FILE = _os.path.join(_OUTPUT_DIR, 'history.json')

def clear_gpu():
    _gc.collect()
    _torch.cuda.empty_cache()
    _free, _total = _torch.cuda.mem_get_info()
    print(f"      🧹 GPU Memory Cleared: {_free/1024**3:.2f} GB free of {_total/1024**3:.2f} GB")
    import sys; sys.stdout.flush()

def save_history(all_histories):
    with open(_HISTORY_FILE, 'w') as f:
        _json.dump(all_histories, f)

def load_history():
    if _os.path.exists(_HISTORY_FILE):
        with open(_HISTORY_FILE, 'r') as f:
            try:
                return _json.load(f)
            except:
                pass
    return {}

def train_model_vsc(model, model_path, train_loader, val_loader, epochs, history_key, all_histories):
    if model_path is not None and _os.path.exists(model_path):
        print(f"✅ Skipping training: {model_path} already exists...")
        model.load_weights(model_path)
    else:
        print(f"🚀 Training '{history_key}'...")
        cb = make_callbacks(model_path) if model_path is not None else []
        hist = model.fit(train_loader, validation_data=val_loader, epochs=epochs, callbacks=cb)
        all_histories[history_key] = hist.history
        if model_path is not None:
            save_history(all_histories)
    
    # Check shape to guarantee no runtime crash on evaluate
    dummy_x, _ = next(iter(val_loader))
    print(f"      [Sanity Check] DataLoader shape: {dummy_x.shape}, Expected: {model.input_shape}")
    
    print(f"📊 Evaluating '{history_key}'...")
    model.evaluate(val_loader)
# ── GPU SETUP ──────────────────────────────────────────────────────────────
# Use Keras 3 with PyTorch backend → runs on your RTX 5060 via CUDA
import os
os.environ["KERAS_BACKEND"] = "torch"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
print("GPU             :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


  📂 Saving mock models locally to: c:\Artificial Neural Networks and Deep Learning\local_test_output
⚠️ LOCAL DRY RUN MODE ACTIVATED ⚠️
=> Overriding Keras Model.fit() to only process 1 batch and 1 epoch!
PyTorch version : 2.7.1+cu118
CUDA available  : True
GPU             : NVIDIA GeForce RTX 5060


c:\Users\shuma\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\cuda\__init__.py:287: UserWarning: 
NVIDIA GeForce RTX 5060 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90 compute_37.
If you want to use the NVIDIA GeForce RTX 5060 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [2]:
import os
import xml.etree.ElementTree as ET
from typing import List

import keras
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
import random

print("Keras version:", keras.__version__)
print("Keras backend:", keras.backend.backend())


Keras version: 3.13.2
Keras backend: tensorflow


In [3]:
_IMAGE_SHAPE = (128, 128)   # H x W used for segmentation
IMG_SIZE = 224           # used for classification
_BATCH_SIZE  = 8            # keep small for 8GB VRAM


# 1. Image Classification
## Preprocessing
Specify the path to the extracted VOC folder below.

In [4]:
path_to_extracted_folder = 'C:/Artificial Neural Networks and Deep Learning/data/VOCtrainval_11-May-2012_2'
path_to_VOC_folder       = path_to_extracted_folder + '/VOCdevkit/VOC2012'


In [5]:
_VOC_LABELS = (
    "aeroplane","bicycle","bird","boat","bottle",
    "bus","car","cat","chair","cow",
    "diningtable","dog","horse","motorbike","person",
    "pottedplant","sheep","sofa","train","tvmonitor",
)

def preprocess_classification_data(voc_root_folder):
    annotation_dir = os.path.join(voc_root_folder, "Annotations")
    image_dir      = os.path.join(voc_root_folder, "JPEGImages")
    dataset = {}
    for xml_file in os.listdir(annotation_dir):
        if not xml_file.endswith(".xml"):
            continue
        tree    = ET.parse(os.path.join(annotation_dir, xml_file))
        root    = tree.getroot()
        fname   = root.find("filename").text
        img_id  = os.path.splitext(fname)[0]
        classes = list({obj.find("name").text for obj in root.findall("object")})
        dataset[img_id] = {"path": os.path.join(image_dir, fname), "classes": classes}
    return dataset


In [6]:
image_classes = preprocess_classification_data(path_to_VOC_folder)
print(f"Total images: {len(image_classes)}")


Total images: 17125


In [7]:
# Example entry
image_classes['2009_003541']


{'path': 'C:/Artificial Neural Networks and Deep Learning/data/VOCtrainval_11-May-2012_2/VOCdevkit/VOC2012\\JPEGImages\\2009_003541.jpg',
 'classes': ['sheep', 'cat']}

Let's plot some random images from the dataset.

In [8]:
print("\n  [CELL 11/47] Data Augmentation check...")
_cell_timer(11)
def show_random_images(dataset, n=4):
    keys = random.sample(list(dataset.keys()), n)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    for ax, key in zip(axes, keys):
        sample = dataset[key]
        img = Image.open(sample["path"])
        ax.imshow(img); ax.axis("off")
        ax.set_title(key + ": " + ", ".join(sample["classes"]), fontsize=8)
    plt.tight_layout(); plt.show()

show_random_images(image_classes)



  [CELL 11/47] Data Augmentation check...
  ⏱  Total elapsed: 0h 00m 05s


C:\Users\shuma\AppData\Local\Temp\ipykernel_10924\3252187318.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Multi-hot encoding & train/val/test split

In [9]:
print("\n  [CELL 13/47] Normalisation check...")
_cell_timer(13)
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

image_paths_cls, labels_cls = [], []
for vals in image_classes.values():
    image_paths_cls.append(vals['path'])
    labels_cls.append(vals['classes'])

mlb = MultiLabelBinarizer(classes=_VOC_LABELS)
y_cls_encoded = mlb.fit_transform(labels_cls)
print(f"Samples: {len(image_paths_cls)}, Label shape: {y_cls_encoded.shape}")



  [CELL 13/47] Normalisation check...
  ⏱  Total elapsed: 0h 00m 05s
Samples: 17125, Label shape: (17125, 20)


In [10]:
train_paths, test_paths, train_labels, test_labels = train_test_split(
    image_paths_cls, y_cls_encoded, test_size=0.2, random_state=42)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels, test_size=0.25, random_state=42)
print(f"Train {len(train_paths)} | Val {len(val_paths)} | Test {len(test_paths)}")


Train 10275 | Val 3425 | Test 3425


### PyTorch DataLoaders
We use PyTorch DataLoaders since Keras 3 can consume them directly on the GPU.

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

class VOCClassificationDataset(Dataset):
    def __init__(self, paths, labels, img_size=224, augment=False):
        self.paths   = paths
        self.labels  = labels.astype(np.float32)
        aug = [T.RandomHorizontalFlip(), T.ColorJitter(0.2, 0.2, 0.1, 0.05)] if augment else []
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            *aug,
            T.ToTensor(),         # → [0,1] float32 CHW
        ])

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img), torch.tensor(self.labels[idx])

# PyTorch DataLoaders (num_workers=0 is safest on Windows)
train_loader = DataLoader(VOCClassificationDataset(train_paths, train_labels, augment=True),
                          batch_size=_BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(VOCClassificationDataset(val_paths,   val_labels),
                          batch_size=_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(VOCClassificationDataset(test_paths,  test_labels),
                          batch_size=_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# Keras 3 can consume PyTorch DataLoaders directly
x_batch, y_batch = next(iter(train_loader))
print("Image batch:", x_batch.shape, "  Labels batch:", y_batch.shape)


Image batch: torch.Size([8, 3, 224, 224])   Labels batch: torch.Size([8, 20])


## Model
### 1.1 Create your own network from scratch

Three iterations:
1. **Baseline CNN** – simple stacked conv blocks
2. **Regularised CNN** – + BatchNorm + Dropout
3. **ResNet-like** – Depthwise separable convs + residual connections

In [12]:
# ─────────────────── 1.1  Custom CNNs from scratch ──────────────────────────
from keras import layers, models, callbacks
import keras
import gc

NUM_CLASSES = len(_VOC_LABELS)

# ── Utility: free GPU memory between models ──────────────────────────────────
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB / {torch.cuda.mem_get_info()[1]/1e9:.2f} GB")

# ── Iteration 1 : Baseline CNN ───────────────────────────────────────────────
def build_baseline_cnn():
    inp = keras.Input(shape=(3, 224, 224))
    x   = layers.Conv2D(32, 3, activation='relu', padding='same', data_format='channels_first')(inp)
    x   = layers.MaxPooling2D(data_format='channels_first')(x)
    x   = layers.Conv2D(64, 3, activation='relu', padding='same', data_format='channels_first')(x)
    x   = layers.MaxPooling2D(data_format='channels_first')(x)
    x   = layers.Conv2D(128, 3, activation='relu', padding='same', data_format='channels_first')(x)
    x   = layers.GlobalAveragePooling2D(data_format='channels_first')(x)
    out = layers.Dense(NUM_CLASSES, activation='sigmoid')(x)
    return keras.Model(inp, out, name="Baseline_CNN")

# ── Iteration 2 : CNN + BatchNorm + Dropout ──────────────────────────────────
def build_regularised_cnn():
    inp = keras.Input(shape=(3, 224, 224))
    x   = inp
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, 3, padding='same', data_format='channels_first')(x)
        x = layers.BatchNormalization(axis=1)(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D(data_format='channels_first')(x)
    x   = layers.GlobalAveragePooling2D(data_format='channels_first')(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(NUM_CLASSES, activation='sigmoid')(x)
    return keras.Model(inp, out, name="Regularised_CNN")

# ── Iteration 3 : Depthwise separable + residual connections ─────────────────
# (lighter filters to fit 8GB VRAM: 32→64→128 instead of 64→128→256)
def res_block(x, filters):
    shortcut = layers.Conv2D(filters, 1, padding='same', data_format='channels_first')(x)
    x = layers.DepthwiseConv2D(3, padding='same', data_format='channels_first')(x)
    x = layers.BatchNormalization(axis=1)(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 1, padding='same', data_format='channels_first')(x)
    x = layers.BatchNormalization(axis=1)(x)
    x = layers.Add()([x, shortcut])
    return layers.ReLU()(x)

def build_resnet_like():
    inp = keras.Input(shape=(3, 224, 224))
    x   = layers.Conv2D(32, 3, padding='same', data_format='channels_first')(inp)
    x   = layers.BatchNormalization(axis=1)(x)
    x   = layers.ReLU()(x)
    for f in [32, 64, 128]:    # reduced from [64,128,256] for 8GB GPU
        x = res_block(x, f)
        x = layers.MaxPooling2D(data_format='channels_first')(x)
    x   = layers.GlobalAveragePooling2D(data_format='channels_first')(x)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(NUM_CLASSES, activation='sigmoid')(x)
    return keras.Model(inp, out, name="ResNet_Like")

print("Model builders defined. Models will be built one-at-a-time to save VRAM.")


Model builders defined. Models will be built one-at-a-time to save VRAM.


In [13]:
print("\n" + "★"*80 + "\n  [CELL 19/47] ★ TRAINING BASELINE CNNs ★\n" + "★"*80)
_cell_timer(19)
# ── Train models ONE AT A TIME (clear GPU memory between them) ──────────────

def make_callbacks(ckpt_name):
    return [
        callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=3, monitor='val_loss'),
        callbacks.ModelCheckpoint(ckpt_name, save_best_only=True, monitor='val_loss'),
    ]

all_histories = {}

# ── V1: Baseline ─────────────────────────────────────────────────────────────
clear_gpu()
model_v1 = build_baseline_cnn()
model_v1.compile(optimizer='adam', loss='binary_crossentropy',
                 metrics=['accuracy', keras.metrics.AUC(multi_label=True, name='auc')])
model_v1.summary()
print("\n=== Training Baseline CNN ===")
train_model_vsc(model_v1, _os.path.join(_OUTPUT_DIR, 'best_cnn_v1.keras'), train_loader, val_loader, 20, '+ v1', all_histories)
del model_v1     # free VRAM

# ── V2: Regularised ──────────────────────────────────────────────────────────
clear_gpu()
model_v2 = build_regularised_cnn()
model_v2.compile(optimizer='adam', loss='binary_crossentropy',
                 metrics=['accuracy', keras.metrics.AUC(multi_label=True, name='auc')])
model_v2.summary()
print("\n=== Training Regularised CNN ===")
train_model_vsc(model_v2, _os.path.join(_OUTPUT_DIR, 'best_cnn_v2.keras'), train_loader, val_loader, 20, '+ v2', all_histories)
del model_v2

# ── V3: ResNet-like (128x128, batch=8 for 8GB GPU) ────────────────────────────
clear_gpu()
v3_train = DataLoader(VOCClassificationDataset(train_paths, train_labels, img_size=224, augment=True),
                      batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
v3_val   = DataLoader(VOCClassificationDataset(val_paths, val_labels, img_size=224),
                      batch_size=8, shuffle=False, num_workers=0, pin_memory=True)
v3_test  = DataLoader(VOCClassificationDataset(test_paths, test_labels, img_size=224),
                      batch_size=8, shuffle=False, num_workers=0, pin_memory=True)
def build_resnet_v3():
    inp = keras.Input(shape=(3, 224, 224))
    x = layers.Conv2D(32, 3, padding='same', data_format='channels_first')(inp)
    x = layers.BatchNormalization(axis=1)(x); x = layers.ReLU()(x)
    for f in [32, 64, 128]:
        x = res_block(x, f)
        x = layers.MaxPooling2D(data_format='channels_first')(x)
    x = layers.GlobalAveragePooling2D(data_format='channels_first')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(NUM_CLASSES, activation='sigmoid')(x)
    return keras.Model(inp, out, name="ResNet_Like")
model_v3 = build_resnet_v3()
model_v3.compile(optimizer='adam', loss='binary_crossentropy',
                 metrics=['accuracy', keras.metrics.AUC(multi_label=True, name='auc')])
model_v3.summary()
print("\n=== Training ResNet-like CNN (128x128, batch=8) ===")
train_model_vsc(model_v3, _os.path.join(_OUTPUT_DIR, 'best_cnn_v3.keras'), train_loader, val_loader, 20, '+ v3', all_histories)
del model_v3
clear_gpu()

import gc, torch
for _ in range(3): gc.collect(); torch.cuda.empty_cache()
clear_gpu()
import sys; sys.stdout.flush()



★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  [CELL 19/47] ★ TRAINING BASELINE CNNs ★
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  ⏱  Total elapsed: 0h 00m 06s
GPU memory free: 7.36 GB / 8.55 GB


Model: "Baseline_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 3, 224, 224)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 224, 224)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 112, 112)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 112, 112)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 56, 56)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 56, 56)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 20)             │         2,580 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 95,828 (374.33 KB)

 Trainable params: 95,828 (374.33 KB)

...

 Trainable params: 28,500 (111.33 KB)

 Non-trainable params: 768 (3.00 KB)


=== Training ResNet-like CNN (128x128, batch=8) ===
✅ Skipping training: c:\Artificial Neural Networks and Deep Learning\local_test_output\best_cnn_v3.keras already exists...
      [Sanity Check] DataLoader shape: torch.Size([8, 3, 224, 224]), Expected: (None, 3, 224, 224)
📊 Evaluating '+ v3'...


c:\Users\shuma\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 74 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 604ms/step - accuracy: 0.1250 - auc: 0.1583 - loss: 0.6855
GPU memory free: 7.36 GB / 8.55 GB
GPU memory free: 7.36 GB / 8.55 GB


In [14]:
print("\n  [CELL 20/47] Visualising CNN results...")
_cell_timer(20)
# ── Compare training curves ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, hist in all_histories.items():
    axes[0].plot(hist['val_loss'], label=label)
    axes[1].plot(hist['val_auc'],  label=label)
axes[0].set(title='Validation Loss', xlabel='Epoch', ylabel='BCE Loss'); axes[0].legend()
axes[1].set(title='Validation AUC',  xlabel='Epoch', ylabel='AUC');      axes[1].legend()
plt.tight_layout(); plt.show()

# Test evaluations were already printed after each model above.
print("All 3 CNN iterations completed.")



  [CELL 20/47] Visualising CNN results...
  ⏱  Total elapsed: 0h 00m 09s
All 3 CNN iterations completed.


C:\Users\shuma\AppData\Local\Temp\ipykernel_10924\3759267962.py:8: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[0].set(title='Validation Loss', xlabel='Epoch', ylabel='BCE Loss'); axes[0].legend()
C:\Users\shuma\AppData\Local\Temp\ipykernel_10924\3759267962.py:9: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[1].set(title='Validation AUC',  xlabel='Epoch', ylabel='AUC');      axes[1].legend()
C:\Users\shuma\AppData\Local\Temp\ipykernel_10924\3759267962.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 1.2 Feature extraction and fine-tuning with Xception

1. Feature extraction with three different dense heads
2. End-to-end frozen base
3. Fine-tune last Xception block with low LR

In [15]:
print("\n" + "★"*80 + "\n  [CELL 22/47] ★ XCEPTION FEATURE EXTRACTION ★\n" + "★"*80)
_cell_timer(22)
# ─────────────────── 1.2  Transfer Learning with Xception ───────────────────
clear_gpu()

XC_SIZE = 224   # reduced from 224 to fit 8GB VRAM
XC_BATCH = 8    # small batch for large Xception model

class VOCDatasetCL(Dataset):
    """channels-last for Xception."""
    def __init__(self, paths, labels, img_size=224, augment=False):
        self.paths  = paths
        self.labels = labels.astype(np.float32)
        aug = [T.RandomHorizontalFlip()] if augment else []
        self.transform = T.Compose([T.Resize((img_size, img_size)), *aug, T.ToTensor()])

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        t   = self.transform(img)            # CHW float [0,1]
        # permute removed for Torch channels-first backend
        t   = t * 2.0 - 1.0                 # → [-1, 1]  (Xception input)
        return t, torch.tensor(self.labels[idx])

xc_train = DataLoader(VOCDatasetCL(train_paths, train_labels, augment=True),
                       batch_size=XC_BATCH, shuffle=True,  num_workers=0, pin_memory=True)
xc_val   = DataLoader(VOCDatasetCL(val_paths,   val_labels),
                       batch_size=XC_BATCH, shuffle=False, num_workers=0, pin_memory=True)
xc_test  = DataLoader(VOCDatasetCL(test_paths,  test_labels),
                       batch_size=XC_BATCH, shuffle=False, num_workers=0, pin_memory=True)

# ── Load Xception base (frozen) ──────────────────────────────────────────────
from keras.applications import Xception

base = Xception(weights='imagenet', include_top=False,
                input_shape=(224, 224, 3))
base.trainable = False

# ── Experiment A: Feature extraction – iterate on dense head ─────────────────
def xc_head(dropout=0.3, hidden=None):
    inp  = keras.Input(shape=(3, 224, 224))
    x    = keras.layers.Permute((2, 3, 1))(inp)

    x    = base(x, training=False)
    x    = layers.GlobalAveragePooling2D()(x)
    x    = layers.Dropout(dropout)(x)
    if hidden:
        x = layers.Dense(hidden, activation='relu')(x)
        x = layers.Dropout(dropout)(x)
    out  = layers.Dense(NUM_CLASSES, activation='sigmoid')(x)
    return keras.Model(inp, out)

cb_xc = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
]

xc_histories = {}
for name, dropout, hidden in [("Xc-A1 (simple)",0.3,None),("Xc-A2 (+dense512)",0.4,512),("Xc-A3 (+dense256)",0.5,256)]:
    clear_gpu()
    m = xc_head(dropout=dropout, hidden=hidden)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy', keras.metrics.AUC(multi_label=True, name='auc')])
    print(f"\n=== {name} ===")
    train_model_vsc(m, None, xc_train, xc_val, 10, name, all_histories)
    if name == "Xc-A1 (simple)":
        best_xc_model = m   # keep best for fine-tuning
    else:
        del m

import gc, torch
for _ in range(3): gc.collect(); torch.cuda.empty_cache()
clear_gpu()
import sys; sys.stdout.flush()



★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  [CELL 22/47] ★ XCEPTION FEATURE EXTRACTION ★
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  ⏱  Total elapsed: 0h 00m 09s
GPU memory free: 7.36 GB / 8.55 GB
GPU memory free: 7.36 GB / 8.55 GB

=== Xc-A1 (simple) ===
🚀 Training 'Xc-A1 (simple)'...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.0000e+00 - auc: 0.2332 - loss: 0.7289WARNING:tensorflow:5 out of the last 5 calls to <function TensorFlowTrainer._make_function.<locals>.multi_step_on_iterator at 0x0000028BA28BA200> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. F

In [16]:
print("\n" + "★"*80 + "\n  [CELL 23/47] ★ XCEPTION FINE TUNING ★\n" + "★"*80)
_cell_timer(23)
# ── Experiment C : Fine-tune last block ──────────────────────────────────────
clear_gpu()
base.trainable = True
for layer in base.layers[:-20]:   # freeze all but last ~20 layers
    layer.trainable = False

best_xc_model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-5),
                      loss='binary_crossentropy',
                      metrics=['accuracy', keras.metrics.AUC(multi_label=True, name='auc')])

print("\n=== Fine-tuning last Xception block ===")
train_model_vsc(best_xc_model, _os.path.join(_OUTPUT_DIR, 'best_xception.keras'), xc_train, xc_val, 10, 'best_xc_model', all_histories)

# Free Xception from GPU for next parts
del best_xc_model, base
clear_gpu()

import gc, torch
for _ in range(3): gc.collect(); torch.cuda.empty_cache()
clear_gpu()
import sys; sys.stdout.flush()



★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  [CELL 23/47] ★ XCEPTION FINE TUNING ★
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  ⏱  Total elapsed: 0h 00m 21s
GPU memory free: 7.36 GB / 8.55 GB

=== Fine-tuning last Xception block ===
✅ Skipping training: c:\Artificial Neural Networks and Deep Learning\local_test_output\best_xception.keras already exists...


c:\Users\shuma\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 27 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


      [Sanity Check] DataLoader shape: torch.Size([8, 3, 224, 224]), Expected: (None, 3, 224, 224)
📊 Evaluating 'best_xc_model'...
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3750 - auc: 0.2161 - loss: 0.5267
GPU memory free: 7.36 GB / 8.55 GB
GPU memory free: 7.36 GB / 8.55 GB


# 2. Image Segmentation
## Preprocessing

In [17]:
# ─────────────────── 2.  Image Segmentation ──────────────────────────────────
from torch.utils.data import Dataset, DataLoader

def get_segmentation_paths(base_dir):
    IMAGE_PATH      = os.path.join(base_dir, 'JPEGImages')
    ANNOTATION_PATH = os.path.join(base_dir, 'SegmentationClass')
    LISTS           = os.path.join(base_dir, 'ImageSets', 'Segmentation')
    img_paths, ann_paths = [], []
    with open(os.path.join(LISTS, 'trainval.txt')) as f:
        names = [l.strip() for l in f if l.strip()]
    for name in names:
        img_paths.append(os.path.join(IMAGE_PATH,      name + '.jpg'))
        ann_paths.append(os.path.join(ANNOTATION_PATH, name + '.png'))
    return img_paths, ann_paths

image_paths, annotation_paths = get_segmentation_paths(path_to_VOC_folder)
print(f"Segmentation samples: {len(image_paths)}")


Segmentation samples: 2913


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_rest, y_train, y_rest = train_test_split(
    image_paths, annotation_paths, train_size=0.6, shuffle=True, random_state=2022)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, train_size=0.6, shuffle=True, random_state=2022)
print(f"Train {len(X_train)} | Val {len(X_val)} | Test {len(X_test)}")


Train 1747 | Val 699 | Test 467


In [19]:
SEG_SIZE = 224

class VOCSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths, size=SEG_SIZE, augment=False):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.size       = size
        self.augment    = augment

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        img  = Image.open(self.img_paths[idx]).convert("RGB").resize((self.size, self.size))
        mask = Image.open(self.mask_paths[idx]).resize((self.size, self.size), Image.NEAREST)

        img  = np.array(img,  dtype=np.float32) / 255.0    # HWC [0,1]
        mask = np.array(mask, dtype=np.uint8)
        mask[mask > 0] = 1                                  # foreground vs background
        mask = mask.astype(np.float32)[..., np.newaxis]     # HW1

        if self.augment and random.random() > 0.5:
            img  = img[:, ::-1, :].copy()
            mask = mask[:, ::-1, :].copy()

        # channels-last → channels-first for torch
        img  = torch.from_numpy(img).permute(2, 0, 1)
        mask = torch.from_numpy(mask).permute(2, 0, 1)
        return img, mask

seg_train_loader = DataLoader(VOCSegDataset(X_train, y_train, augment=True),
                               batch_size=16, shuffle=True,  num_workers=0)
seg_val_loader   = DataLoader(VOCSegDataset(X_val,   y_val),
                               batch_size=16, shuffle=False, num_workers=0)
seg_test_loader  = DataLoader(VOCSegDataset(X_test,  y_test),
                               batch_size=16, shuffle=False, num_workers=0)

imgs, masks = next(iter(seg_train_loader))
print("Seg batch img:", imgs.shape, "mask:", masks.shape)


Seg batch img: torch.Size([16, 3, 224, 224]) mask: torch.Size([16, 1, 224, 224])


Let's visualise some images and their segmentation masks.

In [20]:
# Visualise some images and masks
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img  = imgs[i].permute(1,2,0).numpy()
    msk  = masks[i].permute(1,2,0).numpy().squeeze()
    axes[0,i].imshow(img); axes[0,i].axis('off'); axes[0,i].set_title('Image')
    axes[1,i].imshow(msk, cmap='gray'); axes[1,i].axis('off'); axes[1,i].set_title('Mask')
plt.tight_layout(); plt.show()


C:\Users\shuma\AppData\Local\Temp\ipykernel_10924\2492612468.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Model

We implement a **U-Net** (Encoder-Decoder with skip connections) for foreground/background segmentation.
Metric: **Dice Coefficient** (more meaningful than accuracy on imbalanced pixel maps).

In [21]:
print("\n  [CELL 31/47] Building U-Net model...")
_cell_timer(31)
# ── U-Net model (channels-first for PyTorch backend) ─────────────────────────
def build_unet(img_size=224):
    inp = keras.Input(shape=(3, img_size, img_size))

    # Encoder
    def enc_block(x, f):
        x = layers.Conv2D(f, 3, padding='same', activation='relu', data_format='channels_first')(x)
        x = layers.Conv2D(f, 3, padding='same', activation='relu', data_format='channels_first')(x)
        p = layers.MaxPooling2D(data_format='channels_first')(x)
        return x, p

    c1, p1 = enc_block(inp, 32)
    c2, p2 = enc_block(p1,  64)
    c3, p3 = enc_block(p2, 128)

    # Bottleneck
    b = layers.Conv2D(256, 3, padding='same', activation='relu', data_format='channels_first')(p3)
    b = layers.Conv2D(256, 3, padding='same', activation='relu', data_format='channels_first')(b)

    # Decoder
    def dec_block(x, skip, f):
        x = layers.Conv2DTranspose(f, 2, strides=2, padding='same', data_format='channels_first')(x)
        x = layers.Concatenate(axis=1)([x, skip])
        x = layers.Conv2D(f, 3, padding='same', activation='relu', data_format='channels_first')(x)
        x = layers.Conv2D(f, 3, padding='same', activation='relu', data_format='channels_first')(x)
        return x

    d1 = dec_block(b,  c3, 128)
    d2 = dec_block(d1, c2,  64)
    d3 = dec_block(d2, c1,  32)

    out = layers.Conv2D(1, 1, activation='sigmoid', data_format='channels_first')(d3)
    return keras.Model(inp, out, name="UNet")

# ── Custom Dice metric (backend-agnostic via keras.ops) ──────────────────────
import keras.ops as kops

def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = kops.reshape(kops.cast(y_true, 'float32'), [-1])
    y_pred_f = kops.reshape(y_pred, [-1])
    inter    = kops.sum(y_true_f * y_pred_f)
    return (2. * inter + smooth) / (kops.sum(y_true_f) + kops.sum(y_pred_f) + smooth)

model_unet = build_unet(SEG_SIZE)
model_unet.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', dice_coef]
)
model_unet.summary()



  [CELL 31/47] Building U-Net model...
  ⏱  Total elapsed: 0h 00m 24s


Model: "UNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 3, 224,    │          0 │ -                 │
│ (InputLayer)        │ 224)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 32, 224,   │        896 │ input_layer_7[0]… │
│                     │ 224)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 32, 224,   │      9,248 │ conv2d_18[0][0]   │
│                     │ 224)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, 32, 112,   │          0 │ conv2d_19[0][0]   │
│ (MaxPooling2D)      │ 112)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 64, 112,   │     18,496 │ max_pooling2d_9[… │
│                     │ 112)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 64, 112,   │     36,928 │ conv2d_20[0][0]   │
│                     │ 112)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_10    │ (None, 64, 56,    │          0 │ conv2d_21[0][0]   │
│ (MaxPooling2D)      │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 128, 56,   │     73,856 │ max_pooling2d_10… │
│                     │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 128, 56,   │    147,584 │ conv2d_22[0][0]   │
│                     │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_11    │ (None, 128, 28,   │          0 │ conv2d_23[0][0]   │
│ (MaxPooling2D)      │ 28)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 256, 28,   │    295,168 │ max_pooling2d_11… │
│                     │ 28)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_25 (Conv2D)  │ (None, 256, 28,   │    590,080 │ conv2d_24[0][0]   │
│                     │ 28)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 128, 56,   │    131,200 │ conv2d_25[0][0]   │
│ (Conv2DTranspose)   │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256, 56,   │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 56)               │            │ conv2d_23[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_26 (Conv2D)  │ (None, 128, 56,   │    295,040 │ concatenate[0][0] │
│                     │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_27 (Conv2D)  │ (None, 128, 56,   │    147,584 │ conv2d_26[0][0]   │
│                     │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_1  │ (None, 64, 112,   │     32,832 │ conv2d_27[0][0] 

 Total params: 1,925,601 (7.35 MB)

 Trainable params: 1,925,601 (7.35 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
clear_gpu()
print("\n" + "★"*80 + "\n  [CELL 32/47] ★ TRAINING U-NET SEGMENTOR ★\n" + "★"*80)
_cell_timer(32)
cb_seg = [
    callbacks.EarlyStopping(patience=7, restore_best_weights=True, monitor='val_dice_coef', mode='max'),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=4, monitor='val_dice_coef', mode='max'),
    callbacks.ModelCheckpoint(_os.path.join(_OUTPUT_DIR, 'best_unet.keras'), save_best_only=True,
                              monitor='val_dice_coef', mode='max'),
]

print("=== Training U-Net ===")
train_model_vsc(model_unet, None, seg_train_loader, seg_val_loader, 30, 'model_unet', all_histories)

import gc, torch
for _ in range(3): gc.collect(); torch.cuda.empty_cache()
clear_gpu()
import sys; sys.stdout.flush()


GPU memory free: 7.36 GB / 8.55 GB

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  [CELL 32/47] ★ TRAINING U-NET SEGMENTOR ★
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  ⏱  Total elapsed: 0h 00m 25s
=== Training U-Net ===
🚀 Training 'model_unet'...
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.0045 - dice_coef: 0.3759 - loss: 0.6954 - val_accuracy: 0.2137 - val_dice_coef: 0.3507 - val_loss: 0.6788
      [Sanity Check] DataLoader shape: torch.Size([16, 3, 224, 224]), Expected: (None, 3, 224, 224)
📊 Evaluating 'model_unet'...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 472ms/step - accuracy: 0.2137 - dice_coef: 0.3507 - loss: 0.6788
GPU memory free: 7.36 GB / 8.55 GB


### Visualise predictions vs ground truth

In [23]:
# ── Visualise predictions vs ground truth ─────────────────────────────────────
imgs_t, masks_t = next(iter(seg_test_loader))
preds = model_unet.predict(imgs_t[:4])   # shape (4,1,128,128)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(4):
    img  = imgs_t[i].permute(1,2,0).numpy()
    gt   = masks_t[i].permute(1,2,0).numpy().squeeze()
    pr   = preds[i, 0] if preds.ndim == 4 else preds[i].squeeze() # handle shape
    axes[0,i].imshow(img);           axes[0,i].set_title('Image');      axes[0,i].axis('off')
    axes[1,i].imshow(gt,  cmap='gray'); axes[1,i].set_title('GT mask'); axes[1,i].axis('off')
    axes[2,i].imshow(pr>0.5, cmap='gray'); axes[2,i].set_title('Predicted'); axes[2,i].axis('off')
plt.tight_layout(); plt.show()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 220ms/step


C:\Users\shuma\AppData\Local\Temp\ipykernel_10924\3993929279.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


# 3. Image Detection
## Loading the data

In [24]:
# ─────────────────── 3.  Object Detection ────────────────────────────────────
def preprocess_detection_data(voc_root_folder):
    annotation_dir = os.path.join(voc_root_folder, "Annotations")
    image_dir      = os.path.join(voc_root_folder, "JPEGImages")
    dataset = {}
    for xml_file in os.listdir(annotation_dir):
        if not xml_file.endswith(".xml"): continue
        tree = ET.parse(os.path.join(annotation_dir, xml_file))
        root = tree.getroot()
        fname  = root.find("filename").text
        img_id = os.path.splitext(fname)[0]
        size   = root.find("size")
        W, H   = int(size.find("width").text), int(size.find("height").text)
        boxes, labels = [], []
        for obj in root.findall("object"):
            label = obj.find("name").text
            bb    = obj.find("bndbox")
            xmin  = float(bb.find("xmin").text)
            ymin  = float(bb.find("ymin").text)
            xmax  = float(bb.find("xmax").text)
            ymax  = float(bb.find("ymax").text)
            boxes.append([xmin/W, ymin/H, (xmax-xmin)/W, (ymax-ymin)/H])
            labels.append(label)
        dataset[img_id] = {"boxes": boxes, "labels": labels,
                           "path": os.path.join(image_dir, fname)}
    return dataset

annotations = preprocess_detection_data(path_to_VOC_folder)
print(f"Detection samples: {len(annotations)}")


Detection samples: 17125


Let's explore the resulting dataset.

In [25]:
from matplotlib.patches import Rectangle
from matplotlib.colors import hsv_to_rgb

boxes_per_image = [len(v["boxes"]) for v in annotations.values()]
print(f"Min/Max/Mean boxes: {min(boxes_per_image)} / {max(boxes_per_image)} / {np.mean(boxes_per_image):.2f}")

color_map = {}
def label_to_color(label):
    if label not in color_map:
        h, s, v = (len(color_map) * 0.618) % 1, 0.5, 0.9
        color_map[label] = hsv_to_rgb((h, s, v))
    return color_map[label]

def draw_box(ax, box, text, color):
    x, y, w, h = box
    ax.add_patch(Rectangle((x, y), w, h, lw=2, ec=color, fc="none"))
    ax.text(x, y, text, c="white", size=9, va="bottom",
            bbox=dict(fc=color, pad=1, ec="none"))

sample = annotations["2007_000733"]
print(sample)
fig, ax = plt.subplots(dpi=100)
ax.set(xlim=(0,1), ylim=(1,0), xticks=[], yticks=[], aspect="equal")
ax.imshow(plt.imread(sample["path"]),
          extent=[0, 1, 1, 0])
for box, lbl in zip(sample["boxes"], sample["labels"]):
    draw_box(ax, box, lbl, label_to_color(lbl))
plt.show()


Min/Max/Mean boxes: 1 / 56 / 2.34
{'boxes': [[0.10666666666666667, 0.05747126436781609, 0.5, 0.8229885057471265], [0.2288888888888889, 0.46206896551724136, 0.7666666666666667, 0.5379310344827586]], 'labels': ['person', 'motorbike'], 'path': 'C:/Artificial Neural Networks and Deep Learning/data/VOCtrainval_11-May-2012_2/VOCdevkit/VOC2012\\JPEGImages\\2007_000733.jpg'}


C:\Users\shuma\AppData\Local\Temp\ipykernel_10924\3590277419.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Preprocessing the data

We encode each bounding box into a **7×7 YOLO grid** with shape `(7, 7, 25)`.

In [26]:
print("\n  [CELL 40/47] YOLO grid encoding...")
_cell_timer(40)
GRID_S    = 7
DET_SIZE = 224     # reduced from 224 for 8GB VRAM
C_DET     = 20
class_to_idx = {name: idx for idx, name in enumerate(_VOC_LABELS)}

def encode_annotation(boxes, labels, grid_s=7):
    target = np.zeros((grid_s, grid_s, 5 + C_DET), dtype=np.float32)
    for box, label in zip(boxes, labels):
        ci  = class_to_idx.get(label)
        if ci is None: continue
        rx, ry, rw, rh = box
        xc, yc = rx + rw/2, ry + rh/2
        gx, gy = min(int(xc * grid_s), grid_s-1), min(int(yc * grid_s), grid_s-1)
        if target[gy, gx, 0] == 0:
            target[gy, gx, 0]      = 1.0
            target[gy, gx, 1:5]   = [xc*grid_s - gx, yc*grid_s - gy, rw, rh]
            target[gy, gx, 5+ci]  = 1.0
    return target

img_paths_det, Y_det = [], []
for data in annotations.values():
    if len(data['boxes']) > 0:
        img_paths_det.append(data['path'])
        Y_det.append(encode_annotation(data['boxes'], data['labels']))
Y_det = np.array(Y_det, dtype=np.float32)
print(f"Detection samples: {len(img_paths_det)}, target shape: {Y_det[0].shape}")



  [CELL 40/47] YOLO grid encoding...
  ⏱  Total elapsed: 0h 00m 33s
Detection samples: 17125, target shape: (7, 7, 25)


In [27]:
print("\n  [CELL 41/47] Creating detection DataLoaders...")
_cell_timer(41)
class VOCDetDataset(Dataset):
    def __init__(self, paths, targets, size=DET_SIZE):
        self.paths   = paths
        self.targets = targets
        self.size    = size
        self.tf      = T.Compose([T.Resize((size, size)), T.ToTensor()])

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.tf(img), torch.from_numpy(self.targets[idx])

from sklearn.model_selection import train_test_split
det_img_train, det_img_test, det_y_train, det_y_test = train_test_split(
    img_paths_det, Y_det, test_size=0.2, random_state=42)

det_loader_train = DataLoader(VOCDetDataset(det_img_train, det_y_train),
                               batch_size=8, shuffle=True,  num_workers=0)
det_loader_test  = DataLoader(VOCDetDataset(det_img_test,  det_y_test),
                               batch_size=8, shuffle=False, num_workers=0)

xi, yi = next(iter(det_loader_train))
print("Det batch:", xi.shape, yi.shape)



  [CELL 41/47] Creating detection DataLoaders...
  ⏱  Total elapsed: 0h 00m 33s
Det batch: torch.Size([8, 3, 224, 224]) torch.Size([8, 7, 7, 25])


## Model

Simplified **YOLOv1-style** detector with a custom composite loss.

In [28]:
print("\n  [CELL 43/47] Building YOLO model...")
_cell_timer(43)
clear_gpu()

def build_yolo(grid_s=7, num_boxes=1, num_classes=20, img_size=224):
    inp = keras.Input(shape=(3, img_size, img_size))
    x   = layers.Conv2D(32, 7, strides=2, padding='same', data_format='channels_first')(inp)
    x   = layers.BatchNormalization(axis=1)(x); x = layers.LeakyReLU(0.1)(x)
    x   = layers.MaxPooling2D(2, strides=2, data_format='channels_first')(x)

    x   = layers.Conv2D(64, 3, padding='same', data_format='channels_first')(x)
    x   = layers.BatchNormalization(axis=1)(x); x = layers.LeakyReLU(0.1)(x)
    x   = layers.MaxPooling2D(2, strides=2, data_format='channels_first')(x)

    for f in [128, 256]:
        x = layers.Conv2D(f, 3, padding='same', data_format='channels_first')(x)
        x = layers.BatchNormalization(axis=1)(x); x = layers.LeakyReLU(0.1)(x)
    x   = layers.MaxPooling2D(2, strides=2, data_format='channels_first')(x)

    x   = layers.Conv2D(256, 3, padding='same', data_format='channels_first')(x)
    x   = layers.BatchNormalization(axis=1)(x); x = layers.LeakyReLU(0.1)(x)
    x   = layers.MaxPooling2D(2, strides=2, data_format='channels_first')(x)

    x   = layers.GlobalAveragePooling2D(data_format='channels_first')(x)
    x   = layers.Dense(512, activation='relu')(x)
    x   = layers.Dropout(0.5)(x)

    num_out = grid_s * grid_s * (num_boxes * 5 + num_classes)
    out     = layers.Dense(num_out, activation='sigmoid')(x)
    out     = layers.Reshape((grid_s, grid_s, num_boxes * 5 + num_classes))(out)
    return keras.Model(inp, out, name="YOLO_v1")

model_yolo = build_yolo(GRID_S, 1, C_DET, DET_SIZE)
model_yolo.summary()



  [CELL 43/47] Building YOLO model...
  ⏱  Total elapsed: 0h 00m 33s
GPU memory free: 7.36 GB / 8.55 GB


Model: "YOLO_v1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 3, 224, 224)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_33 (Conv2D)              │ (None, 32, 112, 112)   │         4,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 32, 112, 112)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 32, 112, 112)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 32, 56, 56)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_34 (Conv2D)              │ (None, 64, 56, 56)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 64, 56, 56)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 64, 56, 56)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 64, 28, 28)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_35 (Conv2D)              │ (None, 128, 28, 28)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 128, 28, 28)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 128, 28, 28)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_36 (Conv2D)              │ (None, 256, 28, 28)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 256, 28, 28)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 256, 28, 28)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 256, 14, 14)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_37 (Conv2D)              │ (None, 256, 14, 14)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 256, 14, 14)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 256, 14, 14)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 256, 7, 7)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 512)            │       131,58

 Total params: 1,745,289 (6.66 MB)

 Trainable params: 1,743,817 (6.65 MB)

 Non-trainable params: 1,472 (5.75 KB)

In [29]:
clear_gpu()
print("\n" + "★"*80 + "\n  [CELL 44/47] ★ TRAINING YOLO DETECTOR ★\n" + "★"*80)
_cell_timer(44)
def yolo_loss(y_true, y_pred):
    obj_mask   = y_true[..., 0]
    noobj_mask = 1.0 - obj_mask

    # Coordinate loss (only where object exists)
    xy_loss = kops.sum(
        kops.sum(kops.square(y_true[..., 1:3] - y_pred[..., 1:3]), axis=-1) * obj_mask)
    wh_loss = kops.sum(
        kops.sum(kops.square(
            kops.sqrt(y_true[..., 3:5] + 1e-8) - kops.sqrt(kops.abs(y_pred[..., 3:5]) + 1e-8)
        ), axis=-1) * obj_mask)

    # Confidence loss
    c_loss = (kops.sum(kops.square(obj_mask   - y_pred[..., 0]) * obj_mask) +
              0.5 * kops.sum(kops.square(noobj_mask - y_pred[..., 0]) * noobj_mask))

    # Class loss
    cls_loss = kops.sum(
        kops.sum(kops.square(y_true[..., 5:] - y_pred[..., 5:]), axis=-1) * obj_mask)

    return 5.0 * (xy_loss + wh_loss) + c_loss + cls_loss

model_yolo.compile(optimizer=keras.optimizers.Adam(1e-4), loss=yolo_loss)

print("=== Training YOLO-like detector ===")
cb_det = [callbacks.EarlyStopping(patience=5, restore_best_weights=True),
          callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
          callbacks.ModelCheckpoint(_os.path.join(_OUTPUT_DIR, 'best_yolo.keras'), save_best_only=True)]

train_model_vsc(model_yolo, None, det_loader_train, det_loader_test, 20, 'model_yolo', all_histories)
import gc, torch
for _ in range(3): gc.collect(); torch.cuda.empty_cache()
clear_gpu()
import sys; sys.stdout.flush()


GPU memory free: 7.36 GB / 8.55 GB

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  [CELL 44/47] ★ TRAINING YOLO DETECTOR ★
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  ⏱  Total elapsed: 0h 00m 34s
=== Training YOLO-like detector ===
🚀 Training 'model_yolo'...
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 134.8552 - val_loss: 208.4451
      [Sanity Check] DataLoader shape: torch.Size([8, 3, 224, 224]), Expected: (None, 3, 224, 224)
📊 Evaluating 'model_yolo'...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 208.4451
GPU memory free: 7.36 GB / 8.55 GB


### Visualise detection predictions

In [30]:
print("\n  [CELL 46/47] Visualising detection predictions...")
_cell_timer(46)
# ── Visualise predictions ─────────────────────────────────────────────────────
xi_t, yi_t = next(iter(det_loader_test))
preds_det   = model_yolo.predict(xi_t[:4])

idx_to_class = {v: k for k, v in class_to_idx.items()}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, ax in enumerate(axes):
    img = xi_t[i].permute(1,2,0).numpy()
    ax.imshow(img); ax.set(xlim=(0,1), ylim=(1,0), xticks=[], yticks=[])
    pred = preds_det[i]   # (7,7, 25)
    for gy in range(GRID_S):
        for gx in range(GRID_S):
            cell = pred[gy, gx]
            if cell[0] > 0.3:
                cx = (gx + cell[1]) / GRID_S
                cy = (gy + cell[2]) / GRID_S
                w, h = cell[3], cell[4]
                x0, y0 = cx - w/2, cy - h/2
                cls_idx = int(np.argmax(cell[5:]))
                lbl = idx_to_class.get(cls_idx, str(cls_idx))
                draw_box(ax, [x0, y0, w, h], lbl, label_to_color(lbl))
plt.tight_layout(); plt.show()



  [CELL 46/47] Visualising detection predictions...
  ⏱  Total elapsed: 0h 00m 37s
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step


C:\Users\shuma\AppData\Local\Temp\ipykernel_10924\4038075093.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [31]:
# ── Final Summary ──────────────────────────────────────────────────────────────
elapsed = _time.time() - _global_start
hrs, rem = divmod(int(elapsed), 3600)
mins, secs = divmod(rem, 60)
print(f"\n{'='*80}")
print(f"  ✅ ALL TRAINING COMPLETE")
print(f"  Total runtime: {hrs}h {mins:02d}m {secs:02d}s")
print(f"{'='*80}")


  ✅ ALL TRAINING COMPLETE
  Total runtime: 0h 00m 37s
